In [1]:
import arcpy
import pandas as pd
import os
import traceback
from datetime import datetime
import xarray as xr
import pandas as pd
import numpy as np
import os

#### Create a csv file from the information of the netCDF image and generate points in a file Geodatabase to store them

In [ ]:
count = 0
variables = ["Grid_Point_ID", "Longitude","Latitude","Soil_Moisture","Soil_Moisture_DQX","Surface_Temperature","RFI_Prob", "Confidence_Flags"]
# Dictionary to store monthly data
monthly_data = {month: [] for month in range(1, 13)}
times = []
images_path = r"C:\Users\Bill Tsiakiris\Bill Tsiakiris\Lund\GISM01-Master_Thesis\GIS_Projects\Python files\SMOS_L2"
for netCDF in os.listdir(images_path):
    if netCDF[-3:] == ".nc":
        count += 1
        if count in range(0, 700, 100):
            print(f"Processed {count} images!")
        
        print(f"Working on {netCDF}!")
        image = os.path.join(images_path, netCDF)
        ds = xr.open_dataset(image)
        # ---- Extract timestamp from global attributes ----
        image_time = ds.attrs.get("creation_date")
        image_time = image_time.replace("UTC=", "")
        dt_image_time = datetime.strptime(image_time, "%Y-%m-%dT%H:%M:%S")
        month = dt_image_time.month

        ds = xr.open_dataset(image)
        df = ds[variables].to_dataframe().reset_index(drop=True)
        df['creation_date'] = dt_image_time
        df = df.replace(-999, np.nan)
        
        # ---- Append to correct month ----
        monthly_data[month].append(df)

        ds.close()

excel_path = r"C:\Users\Bill Tsiakiris\Bill Tsiakiris\Lund\GISM01-Master_Thesis\GIS_Projects\Python files\SMOS_L2"
os.makedirs(excel_path, exist_ok=True)

for month, dfs in monthly_data.items():
    if not dfs:
        continue

    monthly_df = pd.concat(dfs, ignore_index=True)
    
    # Filter by Longitude and Latitude
    monthly_df = monthly_df[
        (monthly_df['Longitude'] >= -9.6) & (monthly_df['Longitude'] <= 30) &
        (monthly_df['Latitude'] >= 34.5) & (monthly_df['Latitude'] <= 50)
    ]

    output_file = os.path.join(excel_path, f"SMOS_month_{month:02d}.csv")
    monthly_df.to_csv(output_file, index=False)

    print(f"Written {output_file}")